


---

#  Log Parser & Pipeline d'Ingénierie des Caractéristiques

Ce projet met en place une solution complète pour l'extraction, le nettoyage, et l'ingénierie des caractéristiques (Feature Engineering) à partir de divers fichiers logs de serveurs. Il se compose d'un parseur multi-format robuste et d'un pipeline de préparation de données destiné à l'analyse ou à l'apprentissage automatique.

##  Partie 1 : Log Parser (Parseur Multi-Format)

La classe `LogParser` est conçue pour traiter automatiquement des fichiers logs bruts en détectant leur nature et en structurant leurs informations.

### Fonctionnalités principales

* **Détection automatique** du format de chaque fichier analysé.


* **Support des fichiers compressés** (notamment le format `.gz`).


* **Maintien de l'arborescence** d'entrée reproduite fidèlement dans le dossier de sortie.


* **Exportation flexible** permettant de générer des fichiers CSV intermédiaires par fichier source.


* **Fusion** de l'ensemble des données en un seul fichier CSV consolidé.



### Formats supportés et champs extraits

* **Apache Access (Combined Log) :** Extrait l'`ip`, la `date`, la `method`, l'`url`, le `protocol`, le `status_code`, la `response_size`, le `referer`, et le `user_agent`.


* **Apache Error :** Extrait la `date`, le `module`, le `level`, le `pid`, le `tid`, l'`error_code`, et le `message`.


* **Syslog / Messages :** Extrait la `date`, le `hostname`, le `service`, le `pid`, et le `message`.



---

##  Partie 2 : Pipeline de Nettoyage et Feature Engineering

Une fois les données consolidées, le script applique un pipeline strict de nettoyage et de création de variables pour enrichir le jeu de données final.

### Étapes de Nettoyage

* **Traitement des valeurs manquantes :** Afin de ne pas perdre d'informations potentielles, les valeurs textuelles absentes sont remplacées par la catégorie `unknown`. Les valeurs manquantes pour la taille de réponse (`response_size`) sont initialisées à `0`, et les processus/threads inexistants (`pid` / `tid`) sont notés `-1`.


* **Normalisation temporelle :** Toutes les dates sont standardisées au format ISO 8601 (en UTC).


* **Déduplication :** Les doublons sont supprimés en se basant sur des clés spécifiques au type de log (par exemple : datetime, log_type, source_file, message, etc.).



### Ingénierie des Caractéristiques (Feature Engineering)

Le pipeline génère 34 nouvelles variables pour l'analyse :

* **Variables temporelles :** Encodage cyclique de l'heure (sinus/cosinus) et calcul du temps écoulé depuis la première requête.


* **Variables liées à l'IP :** Calcul du nombre de requêtes par IP, de la durée d'activité, du taux de requêtes, du nombre d'IPs uniques par sous-réseau, du taux d'erreur, et de la diversité des méthodes employées.


* **Variables liées au statut :** Catégorisation des codes HTTP en classes (2xx, 3xx, 4xx, 5xx, etc.).


* **Variables d'URL et Referer :** Calcul de la profondeur du chemin, décompte des paramètres et des caractères spéciaux, et calcul de l'entropie de Shannon pour évaluer la structure de l'URL.


* **Variables liées au User-Agent :** Longueur de la chaîne, calcul d'entropie, et détection de bots via une liste de mots-clés spécifiques (ex: *crawler*, *spider*, *googlebot*).



### Transformations Mathématiques et Exportation

* **Encodage :** Les variables catégorielles (méthode, statut, etc.) sont encodées par rareté (le score étant égal à `1 - fréquence_relative`).


* **Transformation Logarithmique :** Une transformation de type `log(1 + x)` est appliquée sur 19 variables asymétriques pour réduire leur étalement.


* **Standardisation :** Les variables numériques continues subissent une standardisation classique (moyenne = 0, écart-type = 1).


* **Optimisation et Export :** Les types de données (dtypes) sont optimisés (`category`, `string`, `float64`, `int64`) avant l'exportation du DataFrame final nettoyé sous format CSV.